# 1. Import libraries and configure paths

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("RetailCleaned.csv")
OUTPUT_FILE = Path("EcommerceDashboardData.xlsx")
SNAPSHOT_DATE = pd.Timestamp("2011-12-10")
CHURN_DAYS = 90


# 2. Load the SQL-cleaned data

In [ ]:
columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer_ID", "Country", "SourceSheet"
]

df = pd.read_csv(INPUT_FILE, header=None, names=columns, low_memory=False)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Customer_ID"] = pd.to_numeric(df["Customer_ID"], errors="coerce")

print("Dataset shape:", df.shape)
display(df.head())
df.info()


# 3. Prepare eligible transactions

In [ ]:
rfm_data = df.loc[
    df["Customer_ID"].notna()
    & (df["Quantity"] > 0)
    & df["InvoiceDate"].notna()
    & df["Price"].notna()
].copy()

rfm_data["Revenue"] = rfm_data["Quantity"] * rfm_data["Price"]

print("Eligible transaction rows:", len(rfm_data))
print("Unique eligible customers:", rfm_data["Customer_ID"].nunique())


# 4. Calculate RFM metrics

In [ ]:
rfm = (
    rfm_data.groupby("Customer_ID")
    .agg(
        Recency=("InvoiceDate", lambda dates: (SNAPSHOT_DATE - dates.max()).days),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

display(rfm.head())
print("Customer-level RFM shape:", rfm.shape)


# 5. Calculate RFM scores

In [ ]:
# Recency: lower days since purchase receives a higher score
rfm["R_Score"] = pd.qcut(
    rfm["Recency"], q=5, labels=False, duplicates="drop"
)
rfm["R_Score"] = rfm["R_Score"].max() - rfm["R_Score"] + 1

# Frequency: higher purchase frequency receives a higher score
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=5, labels=False, duplicates="drop"
) + 1

# Monetary: higher customer revenue receives a higher score
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"], q=5, labels=False, duplicates="drop"
) + 1

for col in ["R_Score", "F_Score", "M_Score"]:
    rfm[col] = rfm[col].astype("Int64")

display(rfm.head())


# 6. Assign customer segments

In [ ]:
def segment_customer(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    elif row["R_Score"] >= 3 and row["F_Score"] >= 3:
        return "Loyal"
    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New"
    elif row["R_Score"] == 3 and row["F_Score"] <= 2:
        return "Needs Attention"
    else:
        return "Others"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

# Rule-based inactivity flag; this is not predictive churn modelling
rfm["IsChurned"] = rfm["Recency"] > CHURN_DAYS

display(rfm.head())


# 7. Review segment and churn summaries

In [ ]:
segment_summary = (
    rfm.groupby("Segment")
    .agg(
        Customer_Count=("Customer_ID", "nunique"),
        Revenue_Contribution=("Monetary", "sum"),
        Average_Recency=("Recency", "mean")
    )
    .sort_values("Customer_Count", ascending=False)
)

churn_summary = (
    rfm["IsChurned"].value_counts()
    .rename_axis("IsChurned")
    .reset_index(name="Customer_Count")
)

display(segment_summary)
display(churn_summary)


# 8. Prepare transaction-level data for Power BI

In [ ]:
powerbi_data = rfm_data.copy()

powerbi_data["InvoiceMonth"] = (
    powerbi_data["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
)

rfm_columns = [
    "Customer_ID", "Recency", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "Segment", "IsChurned"
]

df_final = powerbi_data.merge(
    rfm[rfm_columns],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

print("Final Power BI dataset shape:", df_final.shape)
display(df_final.head())
